# Huấn Luyện Đầu Nháp EAGLE-3.1 Cho Qwen2.5-7B (Vast.ai A100)

Notebook này cho phép thực thi trực quan pipeline 4 bước:
1. **Chuẩn bị dữ liệu lai (Hybrid Dataset):** 20.000 mẫu ShareGPT + 10.000 mẫu Hóa đơn Tiếng Việt / MC-OCR.
2. **Trích xuất Hidden States:** Trích xuất vector ẩn tầng 28 từ `Qwen/Qwen2.5-7B-Instruct` dạng bfloat16.
3. **Huấn luyện EAGLE-3.1 Head:** 1 tầng Decoder + FC Normalization với Dual Loss (CrossEntropy + 0.5 * Smooth L1).
4. **Đóng gói & Xuất bản:** Chuyển đổi sang `float16` và push lên Hugging Face Hub cho runtime vLLM trên Tesla T4.

In [ ]:
# 1. Kiểm tra cấu hình phần cứng GPU (Kỳ vọng: NVIDIA A100 80GB VRAM)
!nvidia-smi

In [ ]:
# 2. Cài đặt các thư viện phụ thuộc
!pip install -r requirements.txt

In [ ]:
# 3. Bước 1: Gộp và xử lý dữ liệu lai
!python data/prepare_hybrid_dataset.py \
    --sharegpt-path "data/raw/sharegpt.json" \
    --vi-path "data/raw/vietnamese_invoices.json" \
    --output-path "data/processed/hybrid_dataset.json" \
    --sharegpt-limit 20000 \
    --vi-limit 10000

In [ ]:
# 4. Bước 2: Trích xuất Hidden States tầng 28
!python extraction/extract_hidden_states.py \
    --model-id "Qwen/Qwen2.5-7B-Instruct" \
    --dataset-path "data/processed/hybrid_dataset.json" \
    --output-dir "data/features" \
    --target-layer 28 \
    --chunk-size 5000

In [ ]:
# 5. Bước 3: Huấn luyện mạng đầu nháp EAGLE-3.1
!python training/train.py \
    --config "configs/train_config.yaml" \
    --resume

In [ ]:
# 6. Bước 4: Đóng gói chuẩn vLLM float16 và tải lên Hugging Face Hub
# Đặt HF_TOKEN trước khi chạy: export HF_TOKEN="hf_..."
!python export/package_and_upload.py \
    --checkpoint-path "checkpoints/checkpoint_epoch_20.pt" \
    --config-path "configs/eagle3_qwen2.5_7b.json" \
    --output-dir "output/eagle3_qwen2.5_7b" \
    --repo-id "theflyingkhui04/qwen2.5-7b-eagle3.1-speculator" \
    --precision "float16" \
    --upload